In [2]:
import pandas as pd
import numpy as np

In [3]:
sales = pd.read_csv("../../../Data/sales.csv")
products = pd.read_csv("../../../Data/products.csv")
stores = pd.read_csv("../../../Data/stores.csv")
category = pd.read_csv("../../../Data/category.csv")
warranty = pd.read_csv("../../../Data/warranty.csv")

Create a new column Revenue = Quantity × Price.

In [16]:
master_sales = sales.merge(products, on="product_id")
master_sales = master_sales.merge(category, on="category_id")
master_sales = master_sales.merge(stores, on="store_id")
master_sales = master_sales.merge(warranty, on="sale_id")

In [17]:
master_sales["revenue"] = (
    master_sales["quantity"] * master_sales["price"]
)

print(master_sales)

          sale_id   sale_date store_id product_id  quantity  \
0       OID-59217  2023-12-31    ST-55       P-20         1   
1       OID-59218  2023-12-31    ST-55       P-20         1   
2       OID-59219  2023-12-31    ST-55       P-21         1   
3       OID-59220  2023-12-31    ST-55       P-22         1   
4       OID-59221  2023-12-31    ST-55       P-22         1   
...           ...         ...      ...        ...       ...   
30831  OID-358351  2022-08-11    ST-33       P-48         1   
30832  OID-358352  2022-02-06    ST-33       P-49         1   
30833  OID-358353  2022-05-27    ST-33       P-49         1   
30834  OID-358354  2022-03-22    ST-33       P-49         1   
30835  OID-358355  2022-07-19    ST-33       P-49         1   

                        product_name category_id launch_date  price  \
0               Apple Watch Series 5       CAT-5  01-09-2019    399   
1               Apple Watch Series 5       CAT-5  01-09-2019    399   
2                          iPh

Create a Price Category column with values Budget, Mid-Range, Premium, and Luxury based on
Product Price.

In [6]:
def price_category(price):
    if price < 50000:
        return "Budget"
    elif price < 100000:
        return "Mid-Range"
    elif price < 150000:
        return "Premium"
    else:
        return "Luxury"

In [7]:
master_sales["price_category"] = master_sales["price"].apply(price_category)

print(master_sales)


            sale_id   sale_date store_id product_id  quantity  \
0        OID-358365  2021-09-16    ST-33       P-44         1   
1        OID-358510  2022-01-28    ST-33       P-44         1   
2        OID-358525  2021-11-26    ST-33       P-44         1   
3        OID-358591  2021-09-15    ST-33       P-44         1   
4        OID-358601  2022-02-07    ST-33       P-44         1   
...             ...         ...      ...        ...       ...   
1040186  OID-358351  2022-08-11    ST-33       P-48         1   
1040187  OID-358352  2022-02-06    ST-33       P-49         1   
1040188  OID-358353  2022-05-27    ST-33       P-49         1   
1040189  OID-358354  2022-03-22    ST-33       P-49         1   
1040190  OID-358355  2022-07-19    ST-33       P-49         1   

                          product_name category_id launch_date  price  \
0                       iPhone 13 Mini       CAT-4  01-09-2021    699   
1                       iPhone 13 Mini       CAT-4  01-09-2021    699   


Create a Store Performance column using Total Revenue:

Excellent

Good

Average

Poor

In [31]:
def performance_category(revenue):
    if revenue >= 5000000:
        return "Excellent"
    elif revenue >= 3000000:
        return "Good"
    elif revenue >= 1000000:
        return "Average"
    else:
        return "Poor"


In [32]:
master_sales["performance_category"] = master_sales["revenue"].apply(performance_category)

In [38]:
print(master_sales[["revenue", "performance_category"]].head(10))

   revenue performance_category
0      399                 Poor
1      399                 Poor
2      699                 Poor
3      999                 Poor
4      999                 Poor
5      999                 Poor
6      999                 Poor
7     1099                 Poor
8     1099                 Poor
9      329                 Poor


In [39]:
print(master_sales.columns)

Index(['sale_id', 'sale_date', 'store_id', 'product_id', 'quantity',
       'product_name', 'category_id', 'launch_date', 'price', 'category_name',
       'store_name', 'city', 'country', 'claim_id', 'claim_date',
       'claim_status', 'revenue', 'store_contribution',
       'performance_category'],
      dtype='object')


Calculate each Store's contribution (%) to Total Revenue using transform() .

In [18]:
store_revenue = master_sales.groupby("store_name")["revenue"].transform("sum")

total_revenue = master_sales["revenue"].sum()

master_sales["store_contribution"] = (
    store_revenue / total_revenue
) * 100

print(master_sales)

          sale_id   sale_date store_id product_id  quantity  \
0       OID-59217  2023-12-31    ST-55       P-20         1   
1       OID-59218  2023-12-31    ST-55       P-20         1   
2       OID-59219  2023-12-31    ST-55       P-21         1   
3       OID-59220  2023-12-31    ST-55       P-22         1   
4       OID-59221  2023-12-31    ST-55       P-22         1   
...           ...         ...      ...        ...       ...   
30831  OID-358351  2022-08-11    ST-33       P-48         1   
30832  OID-358352  2022-02-06    ST-33       P-49         1   
30833  OID-358353  2022-05-27    ST-33       P-49         1   
30834  OID-358354  2022-03-22    ST-33       P-49         1   
30835  OID-358355  2022-07-19    ST-33       P-49         1   

                        product_name category_id launch_date  price  \
0               Apple Watch Series 5       CAT-5  01-09-2019    399   
1               Apple Watch Series 5       CAT-5  01-09-2019    399   
2                          iPh

In [19]:
stores["store_name"].nunique()

73

Create a final KPI DataFrame containing Store Name, Country, Revenue, Revenue Contribution (%),
Average Quantity, and Performance Category

In [23]:
kpi = master_sales.groupby(["store_name", "country"]).agg(
    revenue=("revenue", "sum"),
    contribution = ("store_contribution", "mean"),
    average_quantity=("quantity", "mean")
).reset_index()

In [40]:
kpi["performance_category"] = kpi["revenue"].apply(performance_category)

In [41]:
print(kpi[["store_name", "country", "revenue", "contribution", "average_quantity", "performance_category"]])

                    store_name      country  revenue  contribution  \
0              Apple Amsterdam  Netherlands   143587      0.665489   
1                 Apple Ankara       Turkey  1863064      8.634820   
2              Apple Barcelona        Spain  4824981     22.362539   
3              Apple Bluewater           UK   120291      0.557518   
4         Apple Champs-Élysées       France   143072      0.663102   
5          Apple Covent Garden           UK   122643      0.568419   
6             Apple Dubai Mall          UAE  4427897     20.522157   
7               Apple Istanbul       Turkey  1793999      8.314721   
8         Apple Kurfürstendamm      Germany   176407      0.817601   
9                   Apple Lyon       France   178296      0.826356   
10  Apple Mall of the Emirates          UAE  4545114     21.065428   
11                 Apple Milan        Italy  1361696      6.311109   
12                Apple Munich      Germany   258549      1.198308   
13             Apple


# Business Insights

 1. Revenue calculation helped identify the sales value generated from each transaction.

 2. Price categories (Budget, Mid-Range, Premium, Luxury) made product segmentation easier for pricing analysis.

 3. Store revenue contribution (%) showed which stores generated the highest share of total company revenue.

 4. KPI report summarized each store's performance using Revenue, Average Quantity, and Performance Category.

 5. Apply(), Transform(), and GroupBy enabled efficient feature engineering and business reporting without manual calculations.